# Early termination: a batch that finishes at different times

Every trajectory is a planar two-body orbit that eventually crosses a
collision or an escape radius and then stops changing. The step kernel
maintains a device-side live count of samples still going; on the GPU arm, a
conditional graph node guards the whole step with that count so a captured
CUDA graph's later replays do no work once every sample is done. The `--host`
arm below runs the identical kernel through eagle's host execution
structure, with the guard evaluated eagerly — no GPU needed.

```bash
python examples/python/early_termination.py --host
```

Needs [hawk](https://github.com/amasat01/hawk) installed next to eagle.

**Colab or Kaggle:** run this first cell (it installs the packages and downloads the helper files this notebook uses). Elsewhere it only checks the imports.

In [1]:
# Colab setup, generated by python -m raptor_docs.nb colab_setup (do not edit).
import os

try:
    import google.colab  # noqa: F401
    HOSTED = True  # a Colab runtime
except ImportError:
    HOSTED = "KAGGLE_KERNEL_RUN_TYPE" in os.environ  # a Kaggle kernel

if HOSTED:
    import importlib.metadata, pathlib, re, shutil, subprocess, sys, urllib.request

    gpu = bool(shutil.which("nvidia-smi")) and subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode == 0
    extra = ""
    if gpu:
        # the CUDA major the runtime already has: a CuPy that is installed, else torch's, else the driver's
        major = None
        for m in ("13", "12"):
            try:
                importlib.metadata.version(f"cupy-cuda{m}x")
                major = m
                break
            except importlib.metadata.PackageNotFoundError:
                pass
        if major is None:
            try:
                import torch
                major = "13" if str(torch.version.cuda).startswith("13") else None
            except ImportError:
                pass
        if major is None:
            smi = re.search(r"CUDA Version: (\d+)", subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
            major = "13" if smi and int(smi.group(1)) >= 13 else "12"
        extra = f"[cuda{major}]"
        print(f"GPU found: installing the {extra} extras to match this runtime.")
    pre = ["--pre"] if os.environ.get("RAPTOR_PRE") else []  # optional: test a release candidate
    ref = os.environ.get("RAPTOR_REF", "main")  # optional: fetch the helper files from this git ref
    if pre or "RAPTOR_REF" in os.environ:
        print(f"Pre-release test: installing with --pre, helper files from {ref}.")
    # our packages with --upgrade, so a runtime that kept an older release gets the current one
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pre, "raptor-core", f"raptor-eagle{extra}", f"raptor-hawk{extra}"])
    base = os.environ.get("RAPTOR_RAW_BASE", "https://raw.githubusercontent.com")  # optional: a local mirror
    top = os.environ.get("RAPTOR_WORKDIR") or ("/kaggle/working" if "KAGGLE_KERNEL_RUN_TYPE" in os.environ else "/content")
    root = pathlib.Path(top) / "eagle"
    for rel in (
        "examples/python/early_termination.py",
    ):
        (root / rel).parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{base}/amasat01/eagle/{ref}/{rel}", root / rel)
    (root / "docs/content/userguide/examples").mkdir(parents=True, exist_ok=True)
    os.chdir(root / "docs/content/userguide/examples")
    if not gpu:
        print("Running on the CPU (no GPU in this runtime). For a GPU: " + ("Settings → Accelerator → GPU T4 x2." if "KAGGLE_KERNEL_RUN_TYPE" in os.environ else "Runtime → Change runtime type → T4 GPU."))

try:
    import eagle, hawk, raptor  # noqa: F401
except ImportError as err:
    raise ImportError(
        "eagle, hawk, raptor not found. Install with `pip install raptor-core raptor-eagle raptor-hawk`. For a GPU, add [cuda12] (or [cuda13]) to raptor-eagle and raptor-hawk."
    ) from err

In [2]:
import sys, pathlib

# These examples live in the repository's own examples/python directory, not in
# the installed package — add it to sys.path so the module below
# imports straight from the checkout.
sys.path.insert(0, str(pathlib.Path("../../../../examples/python").resolve()))
import early_termination as demo
print(demo.__name__, "from examples/python/" + pathlib.Path(demo.__file__).name)

early_termination from examples/python/early_termination.py


The step kernel below is what runs once per sample, per step — the live count and the guard it drives are declared vocabulary, not hidden machinery:

In [3]:
import inspect
print(inspect.getsource(demo.step_kernel))

def step_kernel():
    """Trace the gated RK4 step.

    ``alive``/``alive_next`` are explicit 0/1 planes (``T.Terminated`` is
    read-only, so a terminated sample's own flag has to be carried this way).
    ``age``/``age_next`` ping-pong an increment-while-alive counter
    (``age_next = age + alive``): once a sample's ``alive`` reaches 0, its age
    stops advancing, so the age plane read back at the end IS that sample's
    termination step -- there is no per-replay step-number parameter to read
    it from, because a captured graph replay cannot see a scalar that varies
    per replay (a ``Param`` freezes at capture time).

    ``n_active`` is a one-cell accumulator, seeded to the batch size by the
    caller and never re-zeroed: every launch subtracts exactly the number of
    samples that terminate on THIS step (``still - alive``, zero for a sample
    that stays alive or was already dead), so it decays monotonically to zero
    as the batch finishes. A by-key plane must be a fl

By hand: seed a batch, then let `eagle.deploy` build the kernel with hawk.

In [4]:
import numpy as np
import eagle

state = demo.seed_states(1024, seed=0)
plan = eagle.deploy(demo.step_kernel(), scalar_type="float32", targets=("host",))

Then bind the planes and run one step — this is the one call that
actually launches the kernel, with the live-count guard evaluated
eagerly:

In [5]:
batch = state.shape[1]
x = np.ascontiguousarray(state, dtype=np.float32)
alive = np.ones(batch, dtype=np.float32)
age = np.zeros(batch, dtype=np.float32)
x_next = np.zeros_like(x)
alive_next = np.zeros_like(alive)
age_next = np.zeros_like(age)
n_active = np.full(1, float(batch), dtype=np.float32)

bound = plan.bind(
    x=x, alive=alive, age=age, dt=0.01,
    r_esc2=1.5 ** 2, r_col2=0.2 ** 2,  # the demo's escape/collision radii
    x_next=x_next, alive_next=alive_next, age_next=age_next, n_active=n_active)
bound.launch()
print("samples still alive after one step:", int(alive_next.sum()))

samples still alive after one step: 1024


The pieces above, assembled into one call — building, running and
checking every sample terminated, for reference:

In [6]:
results = demo.run_demo(target="host")

target            : host (manifest exec_targets ['host'])
batch x steps     : 1024 x 400 at dt = 0.01
escape / collide  : r > 1.5 / r < 0.2
samples still active after the horizon : 0 (expected 0)
lifetime histogram (10 bins over the REALISED lifetime range [0, 284], not the full horizon): [0, 0, 0, 449, 386, 75, 53, 30, 20, 11]
replay after all-terminated: state bit-identical = True, count stays 0 = True
positive control (count restored nonzero): step runs = True
host syncs        : 200
wall per step (demo measurement, not a benchmark): 62.4 us


`active_after == 0` is the correctness condition this run checks: every
sample terminated within the horizon. `skip_frozen`/`skip_count_zero` prove
the kernel stops touching the state once the live count hits zero. On a
GPU, the same kernel's **guarded** CUDA graph (one that skips a replay once
the count is zero) is timed against an **unguarded** control (the identical
graph, never skipped) and an eager host-guarded loop — see the
[committed card](https://github.com/amasat01/eagle/blob/main/benchmarks/demo_cards/early_termination_card.md)
for the measured ratios (a million samples over 4000 steps), which this host-only run does not reproduce.